# Session 1 · Loading data

Companion notebook for the **Loading data** section of the slides.

| Part | What we see |
|---|---|
| 1 | One small file hides three loading traps |
| 2 | Predict, then run four loads of `orders.tsv` |
| 3 | Only the dtype mistake fails silently |
| 4 | `read_excel` and `sheet_name` |
| 5 | `json_normalize` and `record_path` |

All files are in the `data/` folder next to this notebook.

In [ ]:
# Excel support needs openpyxl. Run once if it is missing:
# %pip install openpyxl

import json
import pandas as pd

pd.__version__

## 1 · One small file hides three loading traps

`orders.tsv` looks harmless: 3 orders, 6 columns. Before loading it with pandas,
let's look at the **raw bytes** of the first two lines.

In [ ]:
f = "data/orders.tsv"

with open(f, "rb") as fh:
    raw = fh.read()

raw[:80]

Three things to notice in the bytes:

- **Tabs** (`\t`) separate the columns, not commas.
- **Accents**: `H\xe9l\xe8ne` is *Hélène* saved in **latin-1**. In UTF-8, `é` would be two bytes (`\xc3\xa9`).
- **Leading zeros**: `00417` is an identifier, not a number.

> **Goal:** after loading, `order_id` must still read `00417`, and every accent must be correct.

## 2 · Predict what each of the four loads returns

Before running each cell, write down:

| Load | Error? | Shape | dtype of `order_id` |
|---|---|---|---|
| `df1` | | | |
| `df2` | | | |
| `df3` | | | |
| `df4` | | | |

The two loads that may fail are wrapped in `try/except`, so the notebook keeps running and still prints the error.

### Load 1: all defaults

In [ ]:
f = "data/orders.tsv"

try:
    df1 = pd.read_csv(f)
except Exception as e:
    print(type(e).__name__, "->", e)

### Load 2: fix the separator only

In [ ]:
try:
    df2 = pd.read_csv(f, sep="\t")
except Exception as e:
    print(type(e).__name__, "->", e)

Same error: pandas must **decode** the bytes into text before it can split on tabs,
so the separator never gets a chance.

### Load 3: separator and encoding

In [ ]:
df3 = pd.read_csv(f, sep="\t", encoding="latin-1")
print(df3.shape)
df3

In [ ]:
df3.dtypes

Accents are correct, but `order_id` became **417**: pandas saw digits and guessed a number.
No error, no warning.

### Load 4: separator, encoding and dtype

In [ ]:
df4 = pd.read_csv(f, sep="\t", encoding="latin-1",
                  dtype={"order_id": str})
print(df4.shape)
df4

In [ ]:
df4.dtypes

`'00417'` is kept as text!

In [ ]:
df4["order_date"] = pd.to_datetime(
    df4["order_date"],
    format="%Y-%m-%d",
    errors="coerce",   # valori non validi -> NaT invece di eccezione
)

df4.dtypes

### Variant: fix only the encoding

What happens if we get the encoding right but forget `sep`?

In [ ]:
df_variant = pd.read_csv(f, encoding="latin-1")
print(df_variant.shape)
df_variant

In [ ]:
df_variant.columns.tolist()

One single column whose name and values still contain the tabs (`\t`).

## Only the dtype mistake fails silently

| Mistake | What you see | How loud |
|---|---|---|
| wrong `sep` | shape `(3, 1)` | **visible**: `df.shape` gives it away |
| wrong `encoding` | `UnicodeDecodeError` | **loud**: the load stops |
| missing `dtype` | `00417` becomes `417` | **silent**: no error, no warning |

Rule of thumb: pass `dtype=str` for ids, zip codes and phone numbers.

## 4 · `sheet_name` selects one sheet or all of them

`orders.xlsx` has two sheets: **Orders** and **Refunds**.

In [ ]:
fx = "data/orders.xlsx"

pd.read_excel(fx)          # default: the first sheet

In [ ]:
pd.read_excel(fx, sheet_name="Refunds")   # one sheet by name

In [ ]:
sheets = pd.read_excel(fx, sheet_name=None)   # all sheets
type(sheets), list(sheets.keys())

In [ ]:
# A dict {sheet name: DataFrame}: loop over it like any dict
for name, sheet in sheets.items():
    print(f"{name}: {sheet.shape}")

After loading, the DataFrame API is identical to the CSV case, **including the silent trap**:
look at `order_id` above. The Excel file stores `'00417'` as text, yet pandas shows `417`.
The same `dtype` habit applies:

In [ ]:
orders = pd.read_excel(fx, sheet_name="Orders", dtype={"order_id": str})
orders.dtypes

In [ ]:
orders

## 5 · `json_normalize` flattens nested dictionaries

Each order in `orders.json` has a nested `customer` **dict** and a nested list of `items`.

In [ ]:
with open("data/orders.json", encoding="utf-8") as fh:
    data = json.load(fh)

data[0]

`pd.DataFrame` keeps the nesting: whole dicts and lists end up inside cells.

In [ ]:
pd.DataFrame(data)

`json_normalize` flattens nested **dicts** into dotted column names.
Nested **lists** stay unexpanded.

In [ ]:
flat = pd.json_normalize(data)
flat

In [ ]:
flat.columns.tolist()

### `record_path` gives one row per nested item

- `record_path="items"`: one row per item
- `meta=[...]`: parent fields copied onto every row (use a list for nested keys)

In [ ]:
items = pd.json_normalize(data, record_path="items",
                          meta=["order_id", ["customer", "country"], ["customer", "name"]],
                          sep='_')
items

Order 00417 has two items, order 00418 has one: 3 rows in total.
`order_id` stays `'00417'` because JSON stores it as a string.

## 6 · Profile before you change anything

Six calls, always in the same order, on every new dataset. Here on `df4`, the correct load of `orders.tsv`.

| Call | Question it answers |
|---|---|
| `.shape` | How many rows and columns? |
| `.dtypes` | Which type is each column? |
| `.head(10)` | What do real rows look like? |
| `.info()` | Where are the NULLs? How much memory? |
| `.describe()` | Which range do numbers span? |
| `.value_counts()` | Which is the number of unique values? |

In [ ]:
df4.shape

In [ ]:
df4.dtypes

In [ ]:
df4.head(10)

In [ ]:
df4.info()

In [ ]:
df4.describe()

In [ ]:
df4["status"].value_counts()

## Practice: profile three files, answer five questions

Three **real** datasets, one per format. For each one you answer the five questions from the slide:

1. How many rows and columns?
2. Which columns have NULLs, and how many?
3. Which columns need a different dtype?
4. What are the unique `status` values?
5. What is the date range?

| File | What it is | Status column | Date column |
|---|---|---|---|
| `data/velib_stations.csv` | every Vélib bike-sharing station in Greater Paris | `is_installed` | `duedate` |
| `data/paris_roadworks.xlsx` | roadworks that disrupt traffic in Paris | `statut` | `date_debut`, `date_fin` |
| `data/pandas_issues.json` | the 100 latest issues and pull requests of the pandas project on GitHub | `state` | `created_at` |

The questions get harder as you go. Write your code in the empty cell below each question.

**Q1.** Open `data/velib_stations.csv` in binary mode and print its first 150 bytes.
Which character separates the columns?

In [ ]:
# Your code here

**Q2.** Load the file into a DataFrame called `velib`. How many rows and columns does it have?

In [ ]:
# Your code here

**Q3.** Which columns have NULLs, and how many?

In [ ]:
# Your code here

**Q4.** Look at `velib.dtypes` and `velib.head(10)`. Which columns need a different dtype?
Then reload the file so that the two identifier columns, `stationcode` and `code_insee_commune`, are kept as text.

In [ ]:
# Your code here

**Q5.** In this dataset the status column is `is_installed`. What are its unique values, and how many stations have each one?

In [ ]:
# Your code here

**Q6.** How many sheets does `data/paris_roadworks.xlsx` contain, and what are their names?

In [ ]:
# Your code here

**Q7.** Load the sheet into a DataFrame called `works`. How many rows and columns does it have?

In [ ]:
# Your code here

**Q8.** Look at `works.dtypes` and `works.head(10)`. What happened to the postcode column `cp_arrondissement`?
Reload the file so that it is kept as text. Which other columns should not be text?

In [ ]:
# Your code here

**Q9.** Load `data/pandas_issues.json` with the `json` module into a variable called `raw`.
How many records does it contain? Print the first one: which fields are nested?

In [ ]:
# Your code here

**Q10.** Flatten the records with `json_normalize` into a DataFrame called `issues`.
How many rows and columns does it have? Which columns come from the nested dict?

In [ ]:
# Your code here

**Q11.** What are the unique values of `state`, and how many records have each one?
How many of the records are pull requests?

In [ ]:
# Your code here

**Q12.** Build a table with **one row per label**, keeping the issue `number` and `state` on every row. How many rows does it have? Which label is the most frequent?

In [ ]:
# Your code here